In [36]:
from datasets import load_from_disk
import numpy as np
import pickle
import re

In [ ]:
# Load (wrong) Huggingface dataset
data_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_90m/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_old.dataset'
ds = load_from_disk(data_path)
ds

In [ ]:
# Load (fixed) Huggingface dataset
data_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_90m/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10.dataset'
ds = load_from_disk(data_path)
ds

In [ ]:
def _reorder_pair(xs, ys):
    # Stable sort by squared radius without NumPy overhead
    order = sorted(range(len(xs)), key=lambda i: xs[i]*xs[i] + ys[i]*ys[i])
    return [xs[i] for i in order], [ys[i] for i in order]

def reorder_batch(batch):
    rx, ry = batch["rel_x_coord"], batch["rel_y_coord"]
    new_x, new_y = [], []
    append_x, append_y = new_x.append, new_y.append  # tiny micro-opt
    for xs, ys in zip(rx, ry):
        xs_sorted, ys_sorted = _reorder_pair(xs, ys)
        append_x(xs_sorted); append_y(ys_sorted)
    batch["rel_x_coord"] = new_x
    batch["rel_y_coord"] = new_y
    return batch

# Tune these for your machine and IO:
NUM_PROC = 208          # e.g., number of CPU cores
READ_BATCH = 1000     # how many rows processed per function call
WRITE_BATCH = 10000    # how many rows written per Arrow chunk

ds = ds.map(
    reorder_batch,
    batched=True,
    batch_size=READ_BATCH,
    writer_batch_size=WRITE_BATCH,
    num_proc=NUM_PROC,
    desc="Reordering rel_x_coord / rel_y_coord by radius"
)

In [ ]:
item = ds[0]

In [ ]:
item['rel_x_coord']

In [ ]:
item['rel_y_coord']

In [ ]:
data_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_90m/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_new.dataset'
ds.save_to_disk(data_path, num_shards=4000)
print("Saved to:", data_path)

In [ ]:
# Retrieve indices for reordering
with open('/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_loader_idx.pkl', "rb") as f: 
    indices = pickle.load(f)

ds_all_indices = list(range(len(ds)))

# Add test and val indices at end of training indices
seen = set(indices)
to_add = []
for idx in ds_all_indices:
    if idx not in seen:
        to_add.append(idx)
        seen.add(idx)

indices.extend(to_add)

In [ ]:
cache_name = "/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_loader_reidx.arrow"
ds_ordered = ds.select(indices,
                       indices_cache_file_name=cache_name)

In [ ]:
# Materialize the new physical order
ds_ordered = ds_ordered.flatten_indices(num_proc=128, writer_batch_size=10000)

In [ ]:
new_path = data_path.replace('.dataset', '_ordered.dataset')
ds_ordered.save_to_disk(new_path, num_shards=4000)
print("Saved to:", new_path)

In [ ]:
print("All Done!")

In [3]:
# Retrieve indices for reordering
with open('/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_loader_idx.pkl', "rb") as f: 
    indices = pickle.load(f)

In [4]:
train_indices = [i for i in range(len(indices))]

In [ ]:
# Save the split as pickle file
with open(f'/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_ordered_precomputed_split_train.pkl', 'wb') as f:
    pickle.dump(train_indices, f)
print("Saved train split.")

In [ ]:
len(train_indices)

In [ ]:
selected_data = dataset.select(train_indices)

In [ ]:
n_nonzero_tokens = list(selected_data['n_nonzero_tokens'])

output_path = f'/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_ordered_n_nonzero_tokens_train.pkl'
with open(output_path, 'wb') as f:
    pickle.dump(n_nonzero_tokens, f)

In [ ]:
len(n_nonzero_tokens)

In [ ]:
# Save the split as pickle file
with open(f'/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_ordered_n_nonzero_tokens_train.pkl', 'rb') as f:
    n_nonzero_tokens = pickle.load(f)

In [7]:
# Save the split as pickle file
with open(f'/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_split_train.pkl', 'rb') as f:
    train_indices = pickle.load(f)

In [8]:
len(train_indices)

81361647

In [9]:
# Save the split as pickle file
with open(f'/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_n_nonzero_tokens_train.pkl', 'rb') as f:
    n_nonzero_tokens = pickle.load(f)

In [10]:
len(n_nonzero_tokens)

81361647

In [2]:
# Load dataset
dataset = load_from_disk('/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_90m/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_ordered.dataset')

In [9]:
cell_ids = list(dataset['cell_id'])

In [15]:
hst_corpus_70m_dataset_names = [
    'xhb23-1b_1p',
    'xhb24-2b_1p',
    'xhb25-2b_1p',
    'xhb26-2b_1p',
    'xhb27-1b_1p',
    'xhl28-1b_1p',
    'xhl29-1b_1p',
    'xhp30-1b_1p',
    'xhs31-2b_1p',
    'xhh32-1b_1p',
    'xhln33-1b_1p',
    'xhp34-1b_1p',
    'xhp35-1b_1p',
    'xhs36-2b_1p',
    'xhb37-3b_1p',
    'xhc38-4b_1p',
    'xhk39-2b_1p',
    'xhl40-2b_2p',
    'xhl41-2b_1p',
    'xhb42-3b_1p',
    'xht43-2b_1p',
    'xhsc44-6b_1p',
    'xhc45-1b_1p',
    'xhl46-1b_1p',
    'xhp48-1b_1p',
    'xhs49-1b_1p',
    'xho50-1b_1p',
    'xhc51-1b_1p',
    'xhd52-1b_1p',
    'xhl53-1b_1p',
    'xho54-1b_1p',
    'xhc55-3b_1p',
    'xhs62-1b_1p',
    'xhb76-16b_1p',
    'xhk79-4b_1p',
    'xhs1000-39b_1p',
    'xhb1002-AT3-1b_1p',
    'xhb1002-AT4-3b_1p',
    'xhb1002-AT5-1b_1p',
    'xhb1002-AT9-6b_1p',
    'xhb1002-AT10-7b_1p',
    'xhb1002-AT12-1b_1p',
    'xhb1002-AT13-2b_1p',
    'xhb1002-AT14-1b_1p',
    'xhb1002-AT15-10b_1p',
    'xhb1002-AT16-15b_1p',
    'xhb1002-AT17-2b_1p',
    'xhb1002-AT18-2b_1p',
    'xhb1002-AT19-1b_1p',
    'xhb1002-AT20-2b_1p',
    'xhb1002-AT21-2b_1p',
    'xhb1002-AT23-9b_1p',
    'xhb1002-AT101-6b_1p',
    'xhb1003-1b_1p',
    'xhln1004-1b_1p',
    'xhl1005-1b_1p',
    'mhb14-1b_1p',
    'mhc15-2b_1p',
    'mhl16-2b_1p',
    'mhl17-2b_1p',
    'mhm18-2b_1p',
    'mho19-4b_1p',
    'mhp20-2b_1p',
    'mhu21-3b_1p',
    'mhl63-10b_1p',
    'chb56-1b_1p',
    'chl57-2b_1p',
    'chl59-8b_1p',
    'chc61-9b_1p',
    'chr78-11b_1p',
    'chp60-1b_1p',
    'mhh22-3b_1p',
    'shp75-4b_1p',
    'xhb1002-AT6-1-1b_1p',
    'xhb1002-AT6-2-1b_1p',
    'xhb1002-AT6-3-1b_1p',
    'xhb1002-AT6-4-1b_1p',
    'xho47-S1-1b_1p',
    'xho47-S2-1b_1p',
    'xhsc1006-2b_1p',
    'xhc1001-1b_1p',
    'mhl80-8b_1p',
    'xhb81-1b_1p',
    'xhl82-2b_1p',
    'mhb83-1b_1p',
    'xhk1007-CV1-41b_1p',
    'xhk1007-CV2-41b_1p',
    'xhk1007-CV3-41b_1p',
    'xhk1007-CV4-41b_1p',
    'xhk1007-CV5-41b_1p',
    'xhk1007-CV6-41b_1p',
    'xhk1007-CV7-41b_1p',
    'xhk1007-CV8-41b_1p',
    'xhk1007-CV9-41b_1p',
    'xhe1008-1-20b_1p',
    'xhe1008-2-20b_1p',
    'xhe1008-3-20b_1p',
    'xhe1008-4-20b_1p',
    'xhe1008-5-20b_1p',
    'xhe1008-6-20b_1p',
    'xhe1008-7-20b_1p',
    'xhe1008-8-20b_1p',
    'xhe1008-9-20b_1p',
    'xhe1008-10-20b_1p'
]

In [23]:
dataset_ids = [dataset_name.split('-')[0] for dataset_name in hst_corpus_70m_dataset_names]
dataset_ids = [re.sub(r'\D', '', s) for s in dataset_ids]

In [25]:
unique_dataset_ids = list(set(dataset_ids))

In [ ]:
dataset_mask = [
    any(dataset_id == f"{cell_id.split('_')[0]}"
        for dataset_id in unique_dataset_ids)
    for cell_id in cell_ids]
filtered_indices = [
    index for index, value in enumerate(dataset_mask) if value]

In [31]:
len(filtered_indices)

70122697

In [33]:
# Save the split as pickle file
with open(f'/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_ordered_70m_precomputed_split_train.pkl', 'wb') as f:
    pickle.dump(filtered_indices, f)

In [34]:
filtered_dataset = dataset.select(filtered_indices)

In [ ]:
n_nonzero_tokens = list(filtered_dataset['n_nonzero_tokens'])

output_path = f'/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_ordered_70m_n_nonzero_tokens_train.pkl'
with open(output_path, 'wb') as f:
    pickle.dump(n_nonzero_tokens, f)